# 00C Demo Setup — reset and prepare Day 1

Run after configuring a fresh Metadata Lakehouse dedicated to this seven-step Guided Demo. This notebook verifies that all installed metadata tables are empty, previews the exact business assets, requires confirmation, removes six outputs and restores five Day 1 sources.

**Destructive business-data reset: only the eleven named tables in the existing `demo` schemas are in scope.** All demo workspaces must use the same isolated Metadata Lakehouse. 00C never deletes metadata or Governance history; a used Metadata Lakehouse blocks reset. Keep current and previous demo executions stopped until `RESET COMPLETE`.


## 1. Load the shared Fabric configuration

The setup notebook uses the same workspace-local `00_env_config` as the rest of the Guided Demo.


In [1]:
from fabricops_kit import setup_notebook

### Use this when the 00_env_config notebook is in the same workspace as this notebook
context = setup_notebook()

### Advanced you can run the 00_env_config notebook that is in another workspace, 
### make sure you have workspace execution rights to that workspace for this to work
### The caller workspace is detected automatically. Configuration workspace example: "<configuration-workspace-id>"
# Advanced: load 00_env_config from another workspace:
# context = setup_notebook(
#     config_workspace_id="<configuration-workspace-id>",
# )


## 2. Import the FabricOps I/O helpers


In [ ]:
from fabricops_kit import (
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_json,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_warehouse_query,
    read_warehouse_table,
    write_lakehouse_table,
    write_warehouse_table,
)


## Preview and confirm the demo reset

Manually create a schema-enabled Lakehouse named `demo_metadata` exclusively for this walkthrough. Point the existing `Metadata` configuration in every participating demo workspace at that same item and run metadata setup in 00B. 00C and 00_env_config do not create Lakehouse items. Do not run Step 1 before 00C: new metadata tables must still be empty.

For a complete lifecycle restart, use a fresh dedicated Metadata Lakehouse and update all demo configuration copies; old Contracts, activations, Runs and history remain in the previous lakehouse. `setup_metadata_tables()` creates/upgrades tables; it does not clear existing rows. Stop schedules and interactive executions from earlier demo/promoted notebook copies before changing configuration or business data. Do not reuse old promoted copies after switching Metadata.

00C uses the Metadata item selected in 00_env_config. Stop demo executions, then set `DEMO_EXECUTIONS_STOPPED=True`. Add exact IDs of earlier demo/promoted copies sharing these tables to `EXTRA_DEMO_NOTEBOOK_IDS` for live-job checks. Only explicitly confirmed deleted items may skip those checks. Import 02A–02C with their original notebook names.

Business stores may contain other schemas. Reserve these eleven `demo` tables for this walkthrough; other tables, schemas and source files are untouched. Use the lowercase file folder `Files/demo`. Review the preview, confirm `RESET DEMO`, and execute every remaining cell through verification.


In [ ]:
DEMO_EXECUTIONS_STOPPED = False  # Stop current and earlier demo schedules/interactive executions before setting True.
EXTRA_DEMO_NOTEBOOK_IDS = []  # Exact IDs of earlier demo/promoted notebook copies sharing the demo tables.


In [ ]:
# Demo-only validation and physical cleanup using existing FabricOps foundations.
import json
from uuid import UUID
from urllib.error import HTTPError

from pyspark.sql import functions as F

from fabricops_kit.activated_pipeline.shared import FabricNotebookClient
from fabricops_kit.config.metadata_schemas import metadata_table_physical_schema, metadata_table_schema_registry
from fabricops_kit.config.shared import resolve_fabric_context
from fabricops_kit.io.shared import execute_warehouse_sql, resolve_configured_lakehouse_table, resolve_warehouse_name

_DEMO_SOURCES = {
    ('Bronze', 'orders'): ('demo/orders.csv', 120),
    ('Bronze', 'products'): ('demo/products.csv', 8),
    ('Gold', 'order_history'): ('demo/order_history.csv', 180),
    ('Bronze', 'inventory_movements'): ('demo/inventory_day1.csv', 8),
    ('Bronze', 'product_master_updates'): ('demo/products_day1.csv', 8),
}
_DEMO_OUTPUTS = (
    ('Silver', 'curated_orders'), ('Gold', 'customer_summary'),
    ('Silver', 'inventory_movements_append_all'), ('Silver', 'inventory_movements_append_new_only'),
    ('Silver', 'product_master_scd1'), ('Silver', 'product_master_scd2'),
)
_DEMO_NAMES = {'02A_full_refresh_demo', '02B_incremental_append_demo', '02C_scd_demo'}


def _demo_metadata_counts():
    """Read every installed metadata schema without modifying its records or history."""
    counts = {}
    for table, expected in metadata_table_schema_registry().items():
        schema = metadata_table_physical_schema(_demo_config, table)
        try:
            frame = read_lakehouse_table(table, store='Metadata', schema=schema)
        except Exception as exc:
            raise RuntimeError(f'Cannot read {schema}.{table}. Check the demo Metadata mapping and run metadata setup in 00B.') from exc
        actual = {field.name: field.dataType.simpleString() for field in frame.schema}
        incompatible = [field.name for field in expected if actual.get(field.name) != field.dataType.simpleString()]
        if incompatible:
            raise RuntimeError(f'{table} has missing columns or incompatible types: {", ".join(incompatible)}. '
                               'Run the installed package metadata setup in the fresh demo lakehouse, then restart 00C.')
        counts[table] = frame.count()
    return counts


def _demo_require_empty_metadata(counts):
    """Require a fresh lifecycle instead of deleting Contracts, activations or execution history."""
    populated = ', '.join(f'{table}={count}' for table, count in counts.items() if count)
    if populated:
        raise RuntimeError(f'Demo Metadata is not empty ({populated}). No business reset is permitted. '
                           'Stop all previous demo executions, create a fresh dedicated demo Metadata Lakehouse, '
                           'update every demo 00_env_config, run metadata setup in 00B and restart 00C. '
                           'Existing metadata and Governance history are preserved.')


def _demo_assert_source(actual, expected):
    """Compare full row multisets after normalizing Warehouse connector data types."""
    actual = actual.select(*[F.col(field.name).cast(field.dataType).alias(field.name) for field in expected.schema])
    if actual.exceptAll(expected).limit(1).count() or expected.exceptAll(actual).limit(1).count():
        raise RuntimeError('Day 1 source verification failed: persisted content differs from the fixture. Rerun 00C.')


def _demo_check_jobs():
    """Check live jobs; only explicitly verified retired IDs may be confirmed missing."""
    for notebook_id in _demo_notebook_ids:
        if notebook_id in _demo_extra_notebook_ids and notebook_id not in _demo_listed_notebook_ids:
            # A missing jobs endpoint alone does not establish that the notebook was deleted.
            try:
                _demo_client.request('GET', f'/workspaces/{context.workspace_id}/items/{notebook_id}')
            except HTTPError as exc:
                try:
                    error = json.loads(exc.read()) if exc.code == 404 else {}
                except (ValueError, UnicodeDecodeError):
                    error = {}
                if exc.code == 404 and isinstance(error, dict) and error.get('errorCode') == 'ItemNotFound':
                    # Earlier demo schedules and executions must already have been stopped before the configuration switch.
                    continue
                raise RuntimeError(f'Cannot establish whether retired demo notebook {notebook_id} exists. '
                                   'Check workspace/item permissions and Fabric availability before resetting.') from exc
        path = f'/workspaces/{context.workspace_id}/items/{notebook_id}/jobs/instances'
        while path:
            payload = _demo_client.request('GET', path)
            for job in payload['value']:
                if job['status'] not in {'Completed', 'Failed', 'Cancelled', 'Deduped'}:
                    raise RuntimeError(f'Demo notebook {notebook_id} has a non-terminal Fabric job. Stop it before resetting.')
            path = payload.get('continuationUri')


def _demo_remove_outputs():
    """Remove exact configured output objects and their physical publication markers."""
    import notebookutils
    for store, table in _DEMO_OUTPUTS:
        if store == 'Gold':
            gold = context.paths[store]
            execute_warehouse_sql(spark, gold,
                "IF OBJECT_ID(N'demo.customer_summary', N'U') IS NOT NULL DROP TABLE [demo].[customer_summary]; "
                'SELECT 1 AS demo_reset_dropped;', database_name=resolve_warehouse_name(gold, store))
        else:
            _, _, _, path = resolve_configured_lakehouse_table(store, table, 'demo')
            qualified = f'{_demo_silver_catalog}.`{table}`'
            if spark.catalog.tableExists(qualified):
                location_rows = spark.sql(f'DESCRIBE TABLE EXTENDED {qualified}').where("col_name = 'Location'").collect()
                if len(location_rows) != 1 or location_rows[0]['data_type'].rstrip('/') != path.rstrip('/'):
                    raise RuntimeError(f'Catalog location mismatch for {store}.demo.{table}; reset blocked.')
                spark.sql(f'DROP TABLE {qualified}')
            if notebookutils.fs.exists(path) and not notebookutils.fs.rm(path, True):
                raise RuntimeError(f'Could not remove {store}.demo.{table}; check permissions and retry 00C.')
        print(f'Removed output (missing is OK): {store}.demo.{table}')


def _demo_verify_outputs():
    """Fail if any of the six physical outputs still exists."""
    import notebookutils
    for store, table in _DEMO_OUTPUTS:
        if store == 'Gold':
            exists = read_warehouse_query(
                "SELECT COUNT(*) AS n FROM sys.tables t JOIN sys.schemas s ON t.schema_id=s.schema_id "
                "WHERE s.name='demo' AND t.name='customer_summary'", store='Gold').first()['n'] != 0
        else:
            _, _, _, path = resolve_configured_lakehouse_table(store, table, 'demo')
            exists = notebookutils.fs.exists(path) or spark.catalog.tableExists(f'{_demo_silver_catalog}.`{table}`')
        if exists:
            raise RuntimeError(f'Reset incomplete: {store}.demo.{table} remains. Check permissions and rerun 00C.')


### Preview (reset-preview)

Run this cell and review the exact configured assets and metadata counts before confirming.


In [ ]:
# Preview only. No data or metadata is changed by this cell.
_demo_confirmed = False
if context.readiness_status != 'ready':
    raise RuntimeError('Notebook setup is not ready. Correct 00_env_config and rerun 00C.')
_demo_config, _demo_workspace, _ = resolve_fabric_context()
for _key, _kind in [('Bronze', 'lakehouse'), ('Silver', 'lakehouse'), ('Gold', 'warehouse'), ('Metadata', 'lakehouse')]:
    _store = context.paths[_key]
    if _store.kind != _kind or (_kind == 'lakehouse' and not _store.schema_enabled):
        raise RuntimeError(f'{_key} must be a {_kind} with the demo schema configuration (Lakehouses require schemas).')
_metadata = context.paths['Metadata']
_demo_metadata_identity = (_metadata.workspace_id.casefold(), _metadata.item_id.casefold())
for _key in ('Bronze', 'Silver', 'Gold'):
    _store = context.paths[_key]
    if (_store.workspace_id.casefold(), _store.item_id.casefold()) == _demo_metadata_identity:
        raise RuntimeError('The selected Metadata Lakehouse must be separate from the business stores.')

_demo_before = _demo_metadata_counts()
for _table, _count in _demo_before.items():
    print(f'{_table}: {_count} rows; metadata will not be deleted')
_demo_require_empty_metadata(_demo_before)

_demo_client = FabricNotebookClient()
_demo_notebooks = []
_demo_path = f'/workspaces/{context.workspace_id}/notebooks'
while _demo_path:
    _payload = _demo_client.request('GET', _demo_path)
    _demo_notebooks.extend(_payload['value'])
    _demo_path = _payload.get('continuationUri')
_demo_notebook_ids = set()
for _name in _DEMO_NAMES:
    _matches = [item['id'] for item in _demo_notebooks if item['displayName'] == _name]
    if len(_matches) != 1:
        raise RuntimeError(f'Import exactly one {_name} notebook with its original name before running 00C.')
    _demo_notebook_ids.add(_matches[0])
_demo_listed_notebook_ids = {item['id'] for item in _demo_notebooks}
_demo_extra_notebook_ids = {str(UUID(value)) for value in EXTRA_DEMO_NOTEBOOK_IDS}
_demo_notebook_ids.update(_demo_extra_notebook_ids)
# Resolve the Spark catalog name, then validate its storage location before DROP.
_silver = context.paths['Silver']
_silver_workspace_name = _demo_client.request('GET', f'/workspaces/{_silver.workspace_id}')['displayName']
_silver_name = _demo_client.request('GET', f'/workspaces/{_silver.workspace_id}/items/{_silver.item_id}')['displayName']
_demo_silver_catalog = '.'.join('`' + value.replace('`', '``') + '`' for value in (_silver_workspace_name, _silver_name, 'demo'))
_demo_check_jobs()
# Read and materialize all five fixtures before any destructive operation.
_demo_day1 = {}
for (_store, _table), (_file, _count) in _DEMO_SOURCES.items():
    _frame = read_lakehouse_csv(_file, store='Bronze', header=True, inferSchema=True).cache()
    if _frame.count() != _count:
        raise RuntimeError(f'{_file} must contain {_count} rows. Restore the repository fixture and retry.')
    _demo_day1[(_store, _table)] = _frame
print('WARNING: dedicated demo environment only. Stop schedules and interactive executions until verification finishes.')
for _key in ('Bronze', 'Silver', 'Gold', 'Metadata'):
    _store = context.paths[_key]
    print(f'{_key}: workspace={_store.workspace_id}, item={_store.item_id}')
print(f'Checked demo Notebook IDs: {sorted(_demo_notebook_ids)}')
for _store, _table in _DEMO_SOURCES:
    print(f'RESTORE Day 1: {_store}.demo.{_table}')
for _store, _table in _DEMO_OUTPUTS:
    print(f'DROP output: {_store}.demo.{_table}')
print('All metadata tables are empty. Metadata records removed by 00C: 0.')


### Confirm and apply (reset-confirmation / reset-apply)

After reviewing the preview, enter `RESET DEMO` in the next cell and execute it. Then run the application cell. Keep demo executions stopped.


In [ ]:
# After reviewing reset-preview, enter RESET DEMO below and execute this cell, then reset-apply.
# A fresh application consumes this confirmation. No stdin/widget support is required.
RESET_CONFIRMATION = ""


In [ ]:
_demo_confirmed = False
if RESET_CONFIRMATION != 'RESET DEMO':
    raise RuntimeError('Reset cancelled. Review reset-preview, enter RESET DEMO in reset-confirmation and execute it, then apply.')
RESET_CONFIRMATION = ''
if not DEMO_EXECUTIONS_STOPPED:
    raise RuntimeError('Stop current and earlier demo executions, set DEMO_EXECUTIONS_STOPPED=True and rerun the preview.')
_demo_check_jobs()
_demo_require_empty_metadata(_demo_metadata_counts())
try:
    _demo_remove_outputs()
except Exception as exc:
    raise RuntimeError('Demo business reset incomplete. Resolve the cause and restart 00C from the top.') from exc
_demo_confirmed = True


## 3. Read the same "Orders" dataset via CSV, JSON, Parquet, and Excel.

All four files contain the same 120 logical Orders rows. Paths are relative to the configured Lakehouse `Files` area, so `demo/orders.csv` resolves under `Files/demo/orders.csv`.


In [ ]:
orders_csv_df = read_lakehouse_csv(
    "demo/orders.csv",
    store="Bronze",
    
    header=True,
    inferSchema=True,
)

#display(orders_csv_df)


In [ ]:
orders_json_df = read_lakehouse_json(
    "demo/orders.json",
    store="Bronze",
    
)
#display(orders_json_df)


In [ ]:
orders_parquet_df = read_lakehouse_parquet(
    "demo/orders.parquet",
    store="Bronze",
    
)

#display(orders_parquet_df)


In [ ]:
orders_excel_df = read_lakehouse_excel(
    "demo/orders.xlsx",
    store="Bronze",
    
)
#display(orders_excel_df)


## 4. Read and write the demo inputs into the lakehouse and warehouse tables

`products.csv` becomes the Lakehouse lookup table. `order_history.csv` becomes the Warehouse history table.


In [ ]:
if not _demo_confirmed:
    raise RuntimeError('Run the reset preview and confirmation before restoring sources.')

# Write the Orders table into lakehouse 

write_lakehouse_table(
    _demo_day1[('Bronze', 'orders')],
    table_name="orders",
    store="Bronze",
    schema="demo",
    mode="overwrite",
)

#display(orders_csv_df)

# Optional Read the table that is written in the lakehouse to see its written 
orders_table_df = read_lakehouse_table(
    table_name="orders",
    store="Bronze",
    schema="demo",
    
)

#display(orders_table_df)


In [ ]:
if not _demo_confirmed:
    raise RuntimeError('Run the reset preview and confirmation before restoring sources.')

# Read and Write the Product table into lakehouse 

products_df = read_lakehouse_csv(
    "demo/products.csv",
    store="Bronze",
    
    header=True,
    inferSchema=True,
)

#display(products_df)

write_lakehouse_table(
    _demo_day1[('Bronze', 'products')],
    table_name="products",
    store="Bronze",
    schema="demo",
    mode="overwrite",
)

products_table_df = read_lakehouse_table(
    table_name="products",
    store="Bronze",
    schema="demo",
    
)

#display(products_table_df)


In [ ]:
if not _demo_confirmed:
    raise RuntimeError('Run the reset preview and confirmation before restoring sources.')

order_history_df = read_lakehouse_csv(
    "demo/order_history.csv",
    store="Bronze",
    
    header=True,
    inferSchema=True,
)

#display(order_history_df)

write_warehouse_table(
    _demo_day1[('Gold', 'order_history')],
    schema="demo",
    table_name="order_history",
    store="Gold",
    mode="overwrite",
)

# read_warehouse_table() reads the full table. 
order_history_table_df = read_warehouse_table(
    schema="demo",
    table_name="order_history",
    store="Gold",
    
)

#display(order_history_table_df)

# read_warehouse_query() pushes SQL to the Warehouse before returning the result to Spark. 
order_history_sample_df = read_warehouse_query(
    "SELECT TOP 5 * FROM demo.order_history ORDER BY 1",
    store="Gold",
    
)

#display(order_history_sample_df)


## 5. Restore Day 1 for append and SCD demos

Overwrite the two dedicated Bronze sources after the confirmed reset. Run 00C only when starting or restarting the complete Day 1–3 sequence. Do not run it between days.


In [ ]:
if not _demo_confirmed:
    raise RuntimeError('Run the reset preview and confirmation before restoring sources.')

for path, table in [
    ("demo/inventory_day1.csv", "inventory_movements"),
    ("demo/products_day1.csv", "product_master_updates"),
]:
    day1_df = _demo_day1[('Bronze', table)]
    write_lakehouse_table(
        day1_df, table_name=table, store="Bronze", schema="demo", mode="overwrite",
    )
    print(f"Prepared Bronze demo.{table}: {day1_df.count()} Day 1 rows")


## 6. Verify the complete preparation

Compare all five persisted sources to Day 1, verify six outputs are absent and check every metadata table is still empty. No metadata rows are deleted. An exception means preparation is incomplete: resolve its cause and rerun 00C from the top before beginning Step 1.


In [ ]:
if not _demo_confirmed:
    raise RuntimeError('Reset was not confirmed. Run 00C from the top.')
for (_store, _table), _expected in _demo_day1.items():
    _reader = read_warehouse_table if _store == 'Gold' else read_lakehouse_table
    _demo_assert_source(_reader(table_name=_table, store=_store, schema='demo'), _expected)
_demo_verify_outputs()
_demo_check_jobs()
_demo_after = _demo_metadata_counts()
_demo_require_empty_metadata(_demo_after)
for _table, _count in _demo_after.items():
    print(f'{_table}: removed 0; remaining {_count}')
for _frame in _demo_day1.values():
    _frame.unpersist()
_demo_confirmed = False
print('RESET COMPLETE')
